### Test Forward Pass (Final Ranker)

In [1]:
import torch
input_size = 100
x = torch.randn(1000, input_size)

In [2]:
from Ads import FinalRankerMMoE

final_ranker = FinalRankerMMoE(input_size=input_size,
                               layers=2,
                               expert_num=4,
                               expert_dims=[256],
                               task_dims=[[256, 128, 64, 1], [256, 128, 64, 1]],
                               top_k=4)


/Users/paataugrekhelidze/Projects/Recsys/ranking/ads_Moe/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
out = final_ranker(x)
for task in range(len(out)):
    print(task, out[task].shape)

0 torch.Size([1000, 1])
1 torch.Size([1000, 1])


### Test Forward Pass (DLRM towers, early-stage ranker)

In [ ]:
from Ads import DLRMTower

bottom_mlp_layers = [512, 256, 64]
projection_layer = 128
emb_layers = {
    "uid": int(2e6),
    "campaign": 401, # +1 for null at index 400
    "cat1": 10,
    "cat2": 70,
    "cat3": 1829,
    "cat4": 21,
    "cat5": 51,
    "cat6": 30,
    "cat7": 57196,
    "cat8": 11,
    "cat9": 30,
}

sparse_features = [ "uid", 
                   "campaign", 
                   "cat1", 
                   "cat2",
                   "cat3",
                   "cat4",
                   "cat5",
                   "cat6",
                   "cat7",
                   "cat8",
                   "cat9",
                   "last_n_click_campaigns_1D",
                   "last_n_conversion_campaigns_1D"]

model = DLRMTower(
            bottom_mlp_layers=bottom_mlp_layers,
            projection_layer=projection_layer,
            emb_layers=emb_layers,
            sparse_num=len(sparse_features),
            dense_num=0, # no dense features
            device="cpu"
        )
model

/Users/paataugrekhelidze/Projects/Recsys/ranking/ads_Moe/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DLRMTower(
  (embs): ModuleDict(
    (uid): QREmbeddingBag([1415, 1414], [64, 64], mode=mean)
    (campaign): QREmbeddingBag([21, 20], [64, 64], mode=mean)
    (cat1): QREmbeddingBag([4, 3], [64, 64], mode=mean)
    (cat2): QREmbeddingBag([9, 8], [64, 64], mode=mean)
    (cat3): QREmbeddingBag([44, 42], [64, 64], mode=mean)
    (cat4): QREmbeddingBag([6, 4], [64, 64], mode=mean)
    (cat5): QREmbeddingBag([8, 7], [64, 64], mode=mean)
    (cat6): QREmbeddingBag([6, 5], [64, 64], mode=mean)
    (cat7): QREmbeddingBag([240, 239], [64, 64], mode=mean)
    (cat8): QREmbeddingBag([4, 3], [64, 64], mode=mean)
    (cat9): QREmbeddingBag([6, 5], [64, 64], mode=mean)
  )
  (projection): Linear(in_features=78, out_features=128, bias=True)
)

In [2]:
# test dataloader
from datasets import Dataset
import os

filepath = "../../datasets/criteo_attribution_dataset/processed"

train_df = Dataset.from_parquet(os.path.join(filepath, 'train/train.parquet'))


In [3]:
from torch.utils.data import DataLoader
from Ads import AdsDataset
from functools import partial

B = 2048
WORKERS = 0

# Define voc size for user and campaign ids
user_v = emb_layers["uid"]
campaign_v = emb_layers["campaign"]


# Create a partial function with the arguments bound
collate_with_args = partial(AdsDataset._collate_batch, user_v=user_v, campaign_v=campaign_v)


train_dataset = AdsDataset(data= train_df)
train_ld = DataLoader(dataset=train_dataset, batch_size=B, shuffle=False, num_workers=WORKERS, collate_fn=collate_with_args)
total = len(train_df) // B

In [4]:
iterator = iter(train_ld)
sample = next(iterator)
for k in sample.keys():
    print(k, sample[k].shape)

uid torch.Size([2048])
campaign torch.Size([2048])
cat1 torch.Size([2048])
cat2 torch.Size([2048])
cat3 torch.Size([2048])
cat4 torch.Size([2048])
cat5 torch.Size([2048])
cat6 torch.Size([2048])
cat7 torch.Size([2048])
cat8 torch.Size([2048])
cat9 torch.Size([2048])
position_0 torch.Size([2048])
position_1 torch.Size([2048])
position_2 torch.Size([2048])
position_3 torch.Size([2048])
position_4 torch.Size([2048])
position_5 torch.Size([2048])
position_6 torch.Size([2048])
position_7 torch.Size([2048])
position_8 torch.Size([2048])
position_9 torch.Size([2048])
position_10 torch.Size([2048])
position_11 torch.Size([2048])
position_12 torch.Size([2048])
position_13 torch.Size([2048])
position_14 torch.Size([2048])
position_15 torch.Size([2048])
position_16 torch.Size([2048])
position_17 torch.Size([2048])
position_18 torch.Size([2048])
position_19 torch.Size([2048])
position_20 torch.Size([2048])
position_21 torch.Size([2048])
position_22 torch.Size([2048])
position_23 torch.Size([2048])

In [5]:
emb_indices = dict()
emb_offsets = dict()
for f in sparse_features:
    emb_indices[f] = sample[f]
for f in ["last_n_click_campaigns_1D_offset", "last_n_conversion_campaigns_1D_offset"]:
    emb_offsets[f] = sample[f]

In [6]:
import torch
with torch.no_grad():
    out = model(
        x=None,           # dense_num=0, no dense features
        emb_indices=emb_indices,
        emb_offsets=emb_offsets,
    )
print("Output shape:", out.shape)  # expected: (8, 128)

Output shape: torch.Size([2048, 128])


In [ ]:
# now that we have a functional DLRM tower, we can split and represent user and item features separately